# SYNAPSE: state-dependent synthetic supervision for language models

**Qwen2.5-1.5B base + GSM8K. Select an A100 runtime.**

This notebook is for new SYNAPSE runs. To resume historical `llm_qwen_v2`, use the original pinned SynLess checkout and Drive folder; see [migration instructions](https://github.com/akashm776/SYNAPSE/blob/main/docs/MIGRATION.md).

The completed pilot did not establish an overall gain; its Llama gate failed. See [results](https://github.com/akashm776/SYNAPSE/blob/main/docs/QWEN_PILOT_RESULTS.md). The default reproduces the original design, not a new optimized recipe. This is not a LESS/BIDS reproduction or a genuine OT solver.

**Compute:** the original run took about 14 hours for training/development plus 21.5 hours of test evaluation. Inspect preflight resources, but do not confuse step timings with whole-run cost. No Llama launch is automatic.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
REPO = Path("/content/SYNAPSE")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/akashm776/SYNAPSE.git", str(REPO)], check=True)
CODE_REF = ""  # For resume on a new runtime, enter the original printed commit.
if CODE_REF:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REF], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Record this code revision:", COMMIT)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
               "torch==2.9.1", "torchvision==0.24.1", "torchaudio==2.9.1",
               "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)+"[llm,test]",
               "numpy==2.3.5", "accelerate==1.2.1"], check=True)
subprocess.run([sys.executable, "-c",
    "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); "
    "assert 'A100' in torch.cuda.get_device_name(), 'Select an A100 runtime'"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_llm.py"], cwd=REPO, check=True)

## Persistent output and optional model access

Mount Drive. Use a new output folder for a changed code/config/environment. Resume requires the exact original files, revisions, and checkpoints. Never run two sessions against the same folder. Qwen is public; a Hugging Face token is optional here. For future gated models use Colab Secrets named `HF_TOKEN`; never paste credentials into the notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
    del token
except Exception:
    print("No HF_TOKEN secret loaded; public Qwen access will be used.")
CONFIG = REPO / "configs/llm_qwen_a100.json"
OUTPUT = Path("/content/drive/MyDrive/SYNAPSE/runs/qwen_v1")
OUTPUT.mkdir(parents=True, exist_ok=True)
def launch(command):
    subprocess.run([sys.executable, "-m", "synapse.llm.cli", command,
                    "--config", str(CONFIG), "--output", str(OUTPUT)], cwd=REPO, check=True)
print("Config:", CONFIG)
print("Output:", OUTPUT)

## 1. Preflight — required

Downloads pinned-at-first-use model/data revisions, audits splits and token boundaries, tests live higher-order gradients and non-mutating virtual updates, and measures native/auxiliary/meta-step memory on long training examples.

An OOM, nonfinite gradient, missing learning signal, or inadequate memory headroom stops the workflow. Do not silently switch attention kernels, quantize, or shorten sequences inside the same run. No official-test performance is computed here.

In [ ]:
launch("preflight")
preflight = json.loads((OUTPUT / "preflight.json").read_text())
assert preflight["passed"]
print(json.dumps(preflight, indent=2))
audit = json.loads((OUTPUT / "candidate_audit.json").read_text())
print("Eligible partition sizes:", audit["eligible_counts"])
print("Excluded records:", len(audit["excluded"]))

## 2. Train and evaluate development only

Run this after inspecting the preflight. The runner saves native teacher and fresh-learner states, all three learned generators, all seven matched continuation arms, and development metrics. It locks the comparator and checkpoint hashes before reporting.

After a disconnect, rerun setup with the same commit and then this cell; completed work is reused. A partially completed checkpoint interval is replayed deterministically. Do not adjust the experiment after inspecting official-test results.

In [ ]:
launch("run")
print((OUTPUT / "development_lock.json").read_text())

## 3. Official-test report — deliberate separate step

This opens the complete eligible official GSM8K test set for performance reporting. Run only after the development stage is complete and locked. This stage cannot modify generators, comparator choice, or the Llama advancement decision. It evaluates every arm/state/horizon, so decoding is additional GPU work.

In [ ]:
completed = len(list(OUTPUT.glob("seed_*/branches/step_*/*/test_*.json")))
print("Output:", OUTPUT)
print("Completed test evaluations:", completed, "/ 126 for the default config")
# report skips these; an unfinished checkpoint evaluation restarts in full.


In [ ]:
launch("report")
print((OUTPUT / "REPORT.md").read_text())

## Files to share

Share `REPORT.md`, `summary.json`, `development_summary.json`, `development_lock.json`, `preflight.json`, `manifest.json`, `protocol.json`, `candidate_audit.json`, and `command_history.json`. Keep `data.json` and all `.pt` files on Drive for exact resume. No need to publish model weights or credentials.

A failed development gate is a valid research result. Neither influence ranking nor a synthetic mixture alone proves usefulness. The generated objects here are hidden states, not new textual training examples.